# Experiment 15 – Simple Reinforcement Learning Algorithm for an NLP Problem

### Additional Experiment – TensorFlow/Keras

**Aim:** To implement a simple Reinforcement Learning (RL) algorithm for a Natural Language Processing (NLP) problem using Python and TensorFlow/Keras.

### Learning Objectives
- Understand the basic idea of Reinforcement Learning.
- Identify the agent, state, action, reward, and environment in an NLP task.
- Represent simple text inputs as states.
- Train a Q-learning agent using rewards.
- Use the learned policy to choose an appropriate NLP response/action.

## 1. Theory

**Reinforcement Learning** is a machine-learning approach in which an **agent** learns by interacting with an environment and receiving rewards or penalties.

For this simple NLP experiment, the agent learns which response/action is appropriate for a given user message.

### NLP Example

Suppose a user sends:

> “I am very happy today.”

The agent can choose actions such as:

- `positive_response`
- `negative_response`
- `neutral_response`

If the correct action is selected, the agent receives a positive reward. Otherwise, it receives a negative reward.

### RL Components

| RL Component | NLP Meaning |
|---|---|
| Agent | NLP decision-making system |
| State | User message / text category |
| Action | Possible response category |
| Reward | Feedback for the selected action |
| Environment | NLP interaction/task |
| Policy | Rule learned by the agent |

## 2. Problem Definition

We create a small NLP dataset with three simple intent categories:

1. **Greeting** – “hello”, “hi”, “good morning”
2. **Positive** – “I am happy”, “I love this”, “this is wonderful”
3. **Negative** – “I am sad”, “I hate this”, “this is terrible”

The RL agent receives a state representing the detected intent and chooses one of three response actions.

The objective is to learn the action that gives the highest reward for each state.

In [ ]:
# Step 1: Import required libraries
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 3. Create a Simple NLP Environment

We use three text states. Each state represents an intent obtained from a simple NLP classification step.

For this laboratory experiment, the text is manually mapped to an intent so that students can focus on the Reinforcement Learning part.

In [ ]:
# Step 2: Define simple NLP states and actions
states = {
    0: "greeting",
    1: "positive",
    2: "negative"
}

state_examples = {
    0: ["hello", "hi", "good morning"],
    1: ["I am happy", "I love this", "this is wonderful"],
    2: ["I am sad", "I hate this", "this is terrible"]
}

actions = {
    0: "greeting_response",
    1: "positive_response",
    2: "negative_response"
}

print("States:")
for state, name in states.items():
    print(state, "->", name)

print("\nActions:")
for action, name in actions.items():
    print(action, "->", name)

## 4. Define the Reward System

The environment has a simple rule:

- Correct response → **+10 reward**
- Incorrect response → **-5 reward**

For example, if the state is `positive` and the agent chooses `positive_response`, it receives +10.

In [ ]:
# Step 3: Reward function

def get_reward(state, action):
    if state == action:
        return 10
    return -5

print("Reward for correct action:", get_reward(1, 1))
print("Reward for incorrect action:", get_reward(1, 2))

## 5. Q-Learning

**Q-learning** is a Reinforcement Learning algorithm that learns a value called **Q-value** for each state-action pair.

The Q-value represents how useful an action is for a particular state.

The basic update rule is:

```text
Q(s,a) = Q(s,a) + α [R + γ max Q(s',a') - Q(s,a)]
```

Where:

- `α` = learning rate
- `γ` = discount factor
- `R` = reward
- `s` = current state
- `a` = selected action

In this simple problem, the next state is chosen randomly because the focus is on learning the best response for each text intent.

In [ ]:
# Step 4: Initialize the Q-table
num_states = len(states)
num_actions = len(actions)

Q = np.zeros((num_states, num_actions))

learning_rate = 0.1
discount_factor = 0.9
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05
episodes = 1000

print("Initial Q-table:\n")
print(Q)

## 6. Train the RL Agent

The agent follows an **epsilon-greedy strategy**:

- With probability `epsilon`, it explores a random action.
- Otherwise, it chooses the action with the highest Q-value.

During training, epsilon gradually decreases so that the agent moves from exploration toward using what it has learned.

In [ ]:
# Step 5: Train the Q-learning agent
rewards_history = []

epsilon = 1.0

for episode in range(episodes):
    state = random.randrange(num_states)

    # Epsilon-greedy action selection
    if random.random() < epsilon:
        action = random.randrange(num_actions)
    else:
        action = int(np.argmax(Q[state]))

    reward = get_reward(state, action)

    # For this simple environment, use the same state as the next state.
    next_state = state

    best_next_action = np.max(Q[next_state])
    Q[state, action] = Q[state, action] + learning_rate * (
        reward + discount_factor * best_next_action - Q[state, action]
    )

    rewards_history.append(reward)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print("Training completed.")
print("Final epsilon:", round(epsilon, 4))

In [ ]:
# Step 6: Display the learned Q-table
q_table = pd.DataFrame(
    Q,
    index=[states[i] for i in range(num_states)],
    columns=[actions[i] for i in range(num_actions)]
)

print("Learned Q-table:\n")
print(q_table.round(2))

In [ ]:
# Step 7: Plot rewards during training
window = 50
moving_average = np.convolve(
    rewards_history,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(9, 5))
plt.plot(moving_average)
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.title("Q-Learning Training Progress")
plt.show()

## 7. Test the Learned NLP Policy

After training, the agent chooses the action with the highest Q-value for each state.

In [ ]:
# Step 8: Test the learned policy
print("Learned NLP policy:\n")

for state_id in range(num_states):
    best_action = int(np.argmax(Q[state_id]))
    print(
        f"State: {states[state_id]:8s} -> "
        f"Action: {actions[best_action]:18s} | "
        f"Q-value: {Q[state_id, best_action]:.2f}"
    )

## 8. Test with User Text

For this educational example, we use a very simple keyword-based intent detector. In a real NLP application, a trained NLP classifier or language model could identify the intent.

In [ ]:
# Step 9: Simple text-to-intent function

def detect_intent(text):
    text = text.lower()

    positive_words = ["happy", "love", "wonderful", "great", "excellent", "good"]
    negative_words = ["sad", "hate", "terrible", "bad", "angry", "awful"]
    greeting_words = ["hello", "hi", "morning", "hey"]

    if any(word in text for word in positive_words):
        return 1
    if any(word in text for word in negative_words):
        return 2
    if any(word in text for word in greeting_words):
        return 0
    return 0


def get_rl_response(text):
    state = detect_intent(text)
    action = int(np.argmax(Q[state]))
    return states[state], actions[action]

examples = [
    "Hello there",
    "I am very happy today",
    "This is terrible"
]

for text in examples:
    state_name, response_action = get_rl_response(text)
    print(f"Text: {text}")
    print(f"Detected intent: {state_name}")
    print(f"RL action: {response_action}\n")

## 9. Try Your Own NLP Input

Change the text in the following cell and observe the action selected by the RL agent.

In [ ]:
# Step 10: Try your own sentence
user_text = "I love this course"

state_name, response_action = get_rl_response(user_text)

print("User text:", user_text)
print("Detected intent:", state_name)
print("Recommended action:", response_action)

## 10. Student Practice

1. Test at least five different sentences.
2. Add a new intent such as `question`.
3. Add a new action such as `question_response`.
4. Modify the reward values.
5. Change the learning rate and observe the Q-table.
6. Change the number of training episodes.
7. Add more keywords to the intent detector.
8. Explain the role of exploration and exploitation.

### Observation Table

| Input Sentence | Detected Intent | Selected Action | Correct? |
|---|---|---|---|
| __________________ | __________ | __________ | Yes / No |
| __________________ | __________ | __________ | Yes / No |
| __________________ | __________ | __________ | Yes / No |
| __________________ | __________ | __________ | Yes / No |

## 11. Result

Thus, a simple **Reinforcement Learning algorithm for an NLP problem** was successfully implemented using Q-learning. The agent learned suitable response actions for simple text intents using rewards and penalties.

## Viva Questions

1. What is Reinforcement Learning?
2. What is an agent?
3. What is a state in this NLP problem?
4. What is an action?
5. What is a reward?
6. What is Q-learning?
7. What is a Q-table?
8. What is exploration?
9. What is exploitation?
10. Why is reinforcement learning useful for interactive NLP applications?

## 12. Conclusion

In this experiment, students learned the fundamental concepts of Reinforcement Learning and applied Q-learning to a simple NLP decision-making problem. The experiment demonstrated how an agent can learn appropriate actions through rewards and penalties.